# THz Waveguide Dispersion: Worked Reference (typeset)

The five problems solved in `dgs/thz_waveguide_dispersion_relation.py`, with real LaTeX
math instead of Python spelling, plus a glossary at the end. Companion to the HTML
version published as an artifact; this one is the saved, editable, re-runnable copy.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import numpy as np
import pandas as pd
from dgs import thz_waveguide_dispersion_relation as thz
from dgs.thz_griffiths_vocab import vocab_table

a = 0.3e-3
k_c_a = 1.8412
omega_c = thz.C_LIGHT * (k_c_a / a)
omega_op = 1.5 * omega_c
print('Setup complete.')


Setup complete.


## Problem 1 & 2 -- no zero-dispersion point, no two-segment compensation

$$\beta_2(\omega) = \frac{-\omega_c^2}{c(\omega^2-\omega_c^2)^{3/2}} \neq 0 \quad \text{for any finite } \omega$$

The numerator never depends on $\omega$ and is never zero, so $\beta_2$ keeps a fixed
(negative) sign everywhere in the propagating band -- no zero-dispersion frequency, and
no two-segment waveguide link (this same mechanism, two different radii) can cancel its
own dispersion to zero.


In [2]:
ok = thz.verify_gvd_sign_is_fixed()
print(f'beta_2 strictly negative for all omega>omega_c>0, any omega_c: {ok}')


beta_2 strictly negative for all omega>omega_c>0, any omega_c: True


## Problem 3 -- which mode disperses least?

Driving TE11, TM01, TE21 at one shared carrier $\omega_0 = 1.5\,\omega_c^{max}$:

$$\Delta t \approx |\beta_2(\omega_0,\,\omega_c^{mode})| \cdot L \cdot \Delta\omega$$


In [3]:
ranking = thz.rank_modes_by_dispersion(a)
df_rank = pd.DataFrame(ranking['modes']).T
df_rank = df_rank.loc[ranking['ranked_best_to_worst']]
df_rank


,cutoff_THz,m_eff_kg,broadening_ps
TE11,0.292831,2.158894e-39,9.630021
TM01,0.382475,2.819797e-39,20.455975
TE21,0.485761,3.581269e-39,49.135115


## Problem 4 -- does thermal drift matter?

$$a(T) = a_0(1+\alpha\Delta T), \qquad \alpha_{Cu} \approx 17\times10^{-6}\,\text{K}^{-1}$$

Over a 60 K outdoor swing:


In [4]:
thermal = thz.thermal_broadening_shift(a, omega_op)
pd.Series(thermal, name='60K swing, copper wall').to_frame()


,"60K swing, copper wall"
omega_c_frac_shift,-0.001019
broadening_nominal_ps,81.503620
broadening_hot_ps,81.139207
broadening_frac_shift,-0.004471


## Problem 5 -- is the dispersion relation geometry-independent?

$$\omega^2 = c^2(k^2+k_c^2), \qquad k_c^2 \equiv -\frac{\nabla_t^2 f}{f}$$

True for *any* transverse cross-section $f(x,y)$ -- geometry only decides what $k_c$ is,
never the form of the relation. Checked against a real WR-90 rectangular guide below.


In [5]:
ok_general = thz.verify_dispersion_relation_is_geometry_independent()
f_c_wr90 = thz.rectangular_waveguide_cutoff_frequency(1, 0, 22.86e-3, 10.16e-3)
print(f'general proof (any geometry): {ok_general}')
print(f'WR-90 TE10 computed: {f_c_wr90/1e9:.4f} GHz  (published: 6.557 GHz)')


general proof (any geometry): True
WR-90 TE10 computed: 6.5571 GHz  (published: 6.557 GHz)


## Glossary


In [6]:
pd.set_option('display.max_colwidth', 100)
vocab_table()


,symbol,name,meaning,formula,source
0,omega (ω),angular frequency,"How fast the wave oscillates in time, in radians/second (omega = 2*pi*f).",omega = 2πf,dgs/thz_waveguide_dispersion_relation.py
1,omega_c (ωᴄ),cutoff frequency,"The lowest frequency that can actually propagate down the waveguide. Below it, the wave dies out...",omega_c = c·k_c,dgs/cylindrical_waveguide_resonance.py waveguide_cutoff_frequency()
2,k,wavenumber / propagation constant,How many radians of phase the wave accumulates per meter traveled down the guide.,k(omega) = sqrt(omega² - omega_c²) / c,dgs/thz_waveguide_dispersion_relation.py waveguide_wavenumber()
3,k_c,transverse (cutoff) wavenumber,"The wavenumber of the field's cross-sectional pattern, set purely by the waveguide's shape and s...","k_c = j'_(m,n) / a (circular, TE) or k_c = pi*sqrt((m/a)²+(n/b)²) (rectangular)",dgs/cylindrical_waveguide_resonance.py radial_wavenumber(); dgs/thz_waveguide_dispersion_relatio...
4,m_eff,effective photon mass,"A real, nonzero mass a photon behaves as if it has, purely because it's confined in the waveguid...",m_eff = ħ·omega_c / c²,dgs/thz_waveguide_dispersion_relation.py effective_photon_mass()
5,v_phase (v_p),phase velocity,"The speed at which a single crest of the wave appears to move. Exceeds c here, but carries no si...",v_p = omega / k,dgs/thz_waveguide_dispersion_relation.py phase_velocity()
6,v_group (v_g),group velocity,The speed at which energy/information actually travels -- always below c.,v_g = d(omega)/d(k),dgs/thz_waveguide_dispersion_relation.py group_velocity()
7,beta_2 (β₂),group velocity dispersion (GVD),How much the group velocity itself changes with frequency -- the reason a pulse (a bundle of fre...,β₂ = d²k/d(omega)² = -omega_c² / (c·(omega²-omega_c²)^1.5),dgs/thz_waveguide_dispersion_relation.py group_velocity_dispersion()
8,Delta_t,pulse broadening,"How much a pulse's duration grows after traveling distance L, due to GVD.",Δt ≈ |β₂| · L · (bandwidth),dgs/thz_waveguide_dispersion_relation.py thz_pulse_broadening()
9,TE / TM,transverse electric / transverse magnetic mode,TE: the electric field has no component along the guide's axis. TM: the magnetic field has no co...,TE: J'_m(k_c·a)=0 TM: J_m(k_c·a)=0 (circular guide),dgs/cylindrical_waveguide_resonance.py


## Arduino projects for the CSUS THz/benchtop build

Grounded in the actual BOM (`dgs/csus_experiment.py`) and Problem 4's thermal-drift
question above -- not generic Arduino ideas:

1. **Thermal drift logger.** A DS18B20 (or thermocouple + MAX31855) taped to the
   waveguide, logged over a heater/cold-spray-induced temperature sweep, gives real
   data to check against `thermal_broadening_shift`'s -0.10%/-0.45% predictions --
   turns Problem 4 from a calculation into a measurement.
2. **MZM bias controller.** The LiNbO3 modulator's bias point drifts thermally and
   needs to sit at quadrature ($V_\pi/2$) -- an Arduino + DAC (MCP4922) sweeping bias
   voltage while an ADC (ADS1115) reads the photodetector's DC level can find and
   hold quadrature automatically, replacing manual bias-dithering.
3. **RF source frequency sweep + data capture.** Step the RF signal generator (if
   it has a serial/GPIB-to-TTL bridge) and log photodetector output per frequency via
   Arduino's ADC -- builds the actual dispersion curve from bench data instead of
   only the simulated one in this notebook.
4. **Waveguide-length rail (stepper).** For the rectangular/circular waveguide
   comparison (Problem 5), a small stepper-driven rail varying effective path length
   L, logging broadening vs. L, checks the linear-in-L assumption in
   `thz_pulse_broadening` against real hardware.

None of these need a full DAQ card -- an Uno/Nano plus a couple of $5-15 breakout
boards covers all four, consistent with `csus_experiment.py`'s ~$800-2500 total
budget framing.
